# Project Observing Time

This notebook is designed to calculate the total time requested for the various targets within the project, summarized in the different time bins required for the proposal.

In [128]:
import numpy as np

In [401]:
class TargetSet:
    def __init__(self):
        self.survey = None
        self.survey_years = []
        self.survey_semesters = {2026: ['B'], 
                                 2027: ['A', 'B'],
                                 2028: ['A', 'B'],
                                 2029: ['A']}
        self.semester_fraction = []
        self.n_planet_events_per_year = 0
        self.n_stellar_events_per_year = 0
        self.n_bh_events_per_year = 0
        
        self.sinistro_time_per_planet = 0
        self.sinistro_time_per_star = 0
        self.sinistro_time_per_bh = 0
        
        self.floyds_frac = 0.0
        self.floyds_time_per_target = 3.83 # hrs
        
        self.time_per_caustic = 6.0*2.0 # hrs, with two caustic crossings per event
        self.time_per_anomaly = 4.0 # hrs
        self.vis_fraction = 0.0
    
    def calc_total_n_events(self):
        """Calculate the total number of events over the lifetime of the survey operations"""
        self.n_planet_events = 0
        self.n_stellar_events = 0
        self.n_bh_events = 0 
        
        for i,year in enumerate(self.survey_semesters.keys()):
            if len(self.survey_semesters[year]) == 2:
                self.n_planet_events += self.n_planet_events_per_year
                self.n_stellar_events += self.n_stellar_events_per_year
                self.n_bh_events += self.n_bh_events_per_year
                
            elif len(self.survey_semesters[year]) == 1:
                if self.survey_semesters[year][0] == 'A':
                    self.n_planet_events += self.n_planet_events_per_year * self.semester_fraction[1]
                    self.n_stellar_events += self.n_stellar_events_per_year * self.semester_fraction[1]
                    self.n_bh_events += self.n_bh_events_per_year * self.semester_fraction[1]
                else:
                    self.n_planet_events += self.n_planet_events_per_year * self.semester_fraction[0]
                    self.n_stellar_events += self.n_stellar_events_per_year * self.semester_fraction[0]
                    self.n_bh_events += self.n_bh_events_per_year * self.semester_fraction[0]
                
        print('Expect ~'+str(self.n_planet_events_per_year)+' planetary events per year, and '+str(self.n_planet_events)+' over the remaining survey lifetime')
        print('Expect ~'+str(self.n_stellar_events_per_year)+' stellar events per year, and '+str(self.n_stellar_events)+' over the remaining survey lifetime')
        print('Expect ~'+str(self.n_bh_events_per_year)+' stellar remnant events per year, and '+str(self.n_bh_events)+' over the remaining survey lifetime')
      
    def calc_sinistro_time(self):
        """Calculate the time required on the 1m/Sinistro network for different categories of events"""
        
        self.sinistro_time_planet_per_year = self.sinistro_time_per_planet * self.n_planet_events_per_year
        self.sinistro_time_stellar_per_year = self.sinistro_time_per_star * self.n_stellar_events_per_year
        self.sinistro_time_bh_per_year = self.sinistro_time_per_bh * self.n_bh_events_per_year
        time_per_year = self.sinistro_time_planet_per_year + self.sinistro_time_stellar_per_year + self.sinistro_time_bh_per_year
        self.sinistro_time = [time_per_year/2] * 6
        print('Time on 1m/Sinistro network required for planetary events per year = '+str(self.sinistro_time_planet_per_year)+'hrs')
        print('Time on 1m/Sinistro network required for stellar events per year = '+str(self.sinistro_time_stellar_per_year)+'hrs')
        print('Time on 1m/Sinistro network required for stellar remnant events per year = '+str(self.sinistro_time_bh_per_year)+'hrs')
        print('Total Sinistro queue time per semester = ' + str(self.sinistro_time) + 'hrs')
        
    def calc_instrument_time_per_semester(self,inst_name,inst_time_planet_per_year, 
                                          inst_time_stellar_per_year, inst_time_bh_per_year,
                                          planets=True, stellar=True):
        
        inst_time_planet = []
        inst_time_stellar = []
        inst_time_bh = []
        inst_time_total = []
        for i,year in enumerate(self.survey_semesters.keys()):
            for s in self.survey_semesters[year]:
                if s == 'A':
                    if planets:
                        inst_time_planet.append(inst_time_planet_per_year*self.semester_fraction[1])
                    else:
                        inst_time_planet.append(0.0)
                    if stellar:
                        inst_time_stellar.append(inst_time_stellar_per_year*self.semester_fraction[1])
                    else:
                        inst_time_stellar.append(0.0)
                    inst_time_bh.append(inst_time_bh_per_year*self.semester_fraction[1])
                else:
                    if planets:
                        inst_time_planet.append(inst_time_planet_per_year*self.semester_fraction[0])
                    else:
                        inst_time_planet.append(0.0)
                    if stellar:
                        inst_time_stellar.append(inst_time_stellar_per_year*self.semester_fraction[0])
                    else:
                        inst_time_stellar.append(0.0)
                    inst_time_bh.append(inst_time_bh_per_year*self.semester_fraction[0])
                
                time_semester = inst_time_planet[-1]+inst_time_stellar[-1]+inst_time_bh[-1]
                inst_time_total.append(time_semester)
                print('Time on '+inst_name+' in semester '+str(year)+s+': '+str(time_semester))

        return inst_time_planet, inst_time_stellar, inst_time_bh, inst_time_total
                
    def calc_floyds_time(self, planets=True, stellar=True, remnant=True):
        """Calculate the time required for 2m/FLOYDS spectroscopy"""

        ntargets = self.floyds_targets_per_year * self.floyds_frac
        
        self.floyds_bh_events_per_year = int(round(ntargets/3,0))
        print('Number of stellar remnant events bright enough for FLOYDS per year = '+str(self.floyds_bh_events_per_year))
        self.floyds_time_bh_per_year = int(round((self.floyds_bh_events_per_year * self.floyds_time_per_target),0))
        print('Time on 2m/FLOYDS network required for stellar remnant events per year = '+str(self.floyds_time_bh_per_year)+'hrs')

        self.floyds_planet_events_per_year = int(round(ntargets/3,0))
        print('Number of planetary events bright enough for FLOYDS per year = '+str(self.floyds_planet_events_per_year))
        self.floyds_time_planet_per_year = int(round((self.floyds_planet_events_per_year * self.floyds_time_per_target),0))
        print('Time on 2m/FLOYDS network required for planetary events per year = '+str(self.floyds_time_planet_per_year)+'hrs')
        
        self.floyds_stellar_events_per_year = int(round(ntargets/3,0))
        print('Number of stellar events bright enough for FLOYDS per year = '+str(self.floyds_stellar_events_per_year))
        self.floyds_time_stellar_per_year = int(round((self.floyds_stellar_events_per_year * self.floyds_time_per_target),0))
        print('Time on 2m/FLOYDS network required for stellar events per year = '+str(self.floyds_time_stellar_per_year)+'hrs')
            
        self.floyds_time_planet = []
        self.floyds_time_stellar = []
        self.floyds_time_bh = []
        self.floyds_time = []
        for i,year in enumerate(self.survey_years):
            time_year = self.floyds_time_bh_per_year
            if planets:
                time_year += self.floyds_time_planet_per_year
            if stellar:
                time_year += self.floyds_time_stellar_per_year
                
            self.floyds_time.append(time_year)
            print('Total time on 2m/FLOYDS in year '+str(year)+': '+str(time_year))

        (time_planet,time_stellar,time_bh,time_total) = self.calc_instrument_time_per_semester('2m/FLOYDS',
                                                                                    self.floyds_time_planet_per_year, 
                                                                                    self.floyds_time_stellar_per_year, 
                                                                                    self.floyds_time_bh_per_year,
                                                                                    planets=planets,
                                                                                    stellar=stellar)
        self.floyds_time_planet = time_planet
        self.floyds_time_stellar = time_stellar
        self.floyds_time_bh = time_bh
        self.floyds_time = time_total

    def calc_2m_imagers_time(self, ntargets_per_year, caustic_crossing_frac=0.2):
        """Estimate of time required in hours on the 2m/MusCAT or 2m/Spectral network to 
        observe caustic crossings for stellar & planetary binary events in RAPID RESPONSE MODE

        Assume that RR-mode will only be needed for caustic_crossing_frac% of HM events (i.e. caustic crossing only)
        """

        # Assume that RR-mode will only be needed for 10% of HM events (i.e. caustic crossing only)
        print('RR-mode needed for ' + str(caustic_crossing_frac*100) + '% of ' + str(ntargets_per_year)) 
        print('Visibility fraction = ' + str(self.vis_fraction_2m))
        ntargets_per_year = int(round((ntargets_per_year * caustic_crossing_frac),0))
        
        self.time_2m_imagers = []
        for i,year in enumerate(self.survey_years):

            # Observe for a fraction of 12hr total based on visibility
            time_year = ntargets_per_year * self.vis_fraction_2m * 12
        
            print('Total time on 2m/imagers in RR mode in year '+str(year)+': '+str(time_year))

            # Assume a 50% - 50% split of the time over A, B semesters, since the semester break is in the middle of the 
            # Bulge visibility season
            print('Total time on 2m/imagers in RR mode in '+str(year)+'A semester: '+str(round(time_year*0.5, 0)))
            print('Total time on 2m/imagers in RR mode in '+str(year)+'B semester: '+str(round(time_year*0.5, 0)))
            self.time_2m_imagers.append(time_year)
            self.time_2m_imagers.append(time_year)
            
    def calc_1m_rr_time(self, ntargets_per_year, caustic_crossing_frac=0.2):
        """Time required in Rapid Reponse mode from the 1m network is equivalent to the fraction of the time 
        dedicated to observing anomalies and caustic crossings that isn't already covered by the 2m. 
        
        It is only calculated for the planetary and stellar binary events to be followed from each survey. 
        Assumes that RR-mode will only be needed for 10% of HM events (i.e. caustic crossing only)
        """
        ntargets_per_year = int(round((ntargets_per_year * caustic_crossing_frac),0))
        
        self.time_rr_1m_imagers = []
        for i,year in enumerate(self.survey_years):

            # Observe for a fraction of 12hr total based on visibility
            time_year = ntargets_per_year * self.vis_fraction_1m * 12
        
            print('Total time on 1m/imagers in RR mode in year '+str(year)+': '+str(time_year))

            # Assume a 50% - 50% split of the time over A, B semesters, since the semester break is in the middle of the 
            # Bulge visibility season
            print('Total time on 1m/imagers in RR mode in '+str(year)+'A semester: '+str(round(time_year*0.5, 0)))
            print('Total time on 1m/imagers in RR mode in '+str(year)+'B semester: '+str(round(time_year*0.5, 0)))
            self.time_rr_1m_imagers.append(time_year)
            self.time_rr_1m_imagers.append(time_year)
                
    def calc_time_totals(self):
        print('Sinistro queue ' , self.sinistro_time)
        print('Sinistro RR ',self.time_rr_1m_imagers)
        print('2m RR ', self.time_2m_imagers)
        print('FLOYDS ', self.floyds_time)
        
        # Calculate time required over all survey semesters
        self.sinistro_queue_time_total = np.array(self.sinistro_time).sum()
        self.sinistro_rr_time_total = np.array(self.time_rr_1m_imagers).sum()
        self.floyds_time_total = np.array(self.floyds_time).sum()
        self.time_2m_imagers_total = np.array(self.time_2m_imagers).sum()
        
        self.sinistro_1m_queue = np.zeros(len(self.sinistro_time))
        s = -1
        for i,year in enumerate(self.survey_semesters.keys()):
            annual_semesters = self.survey_semesters[year]
            
            for sem in annual_semesters:
                s += 1
                self.sinistro_1m_queue[s] = self.sinistro_time[s]
                print('Semester '+str(year)+sem+' Sinistro total '+str(self.sinistro_time[i]))
                print(' -> Sinistro queue mode '+str(self.sinistro_1m_queue[s]))
                print(' -> Sinistro RR mode '+str(self.time_rr_1m_imagers[s]))

        
        # Calculate time required per semester in each survey year
        # LCO Semesters run A: Feb 1 - July 31 and B: Aug 1 - Jan 31
        # semester_fraction refers to the ratio of the annual target count that 
        # expected to be found in the [B, A] semesters. 
        # The Key Project proposals run from 2023B - 2026A inclusive
        
        print('\n')
        print('Sinistro queue time total over project: '+str(self.sinistro_queue_time_total)+'hrs')
        print('Sinistro queue time total over project: '+str(self.sinistro_rr_time_total)+'hrs')
        print('FLOYDS time total over project: '+str(self.floyds_time_total)+'hrs')
        print('2m imagers RR-time total over project: '+str(self.time_2m_imagers_total)+'hrs')
        
    def calc_time_per_semester(self, annual_time):
        
        key_project_semesters = {}
        
        for i,year in enumerate(self.survey_semesters.keys()):
            annual_semesters = self.survey_semesters[year]
            
            Atime = annual_time[i] * self.semester_fraction[1]
            Btime = annual_time[i] * self.semester_fraction[0]
            
            semester_time = {}
            for s in annual_semesters:
                if s == 'A':
                    semester_time[s] = Atime
                else:
                    semester_time[s] = Btime
                    
            key_project_semesters[year] = semester_time
            print('Year '+str(year)+': '+repr(key_project_semesters[year]))
        
        return key_project_semesters
        

## Microlensing Observations

This section draws on estimates of the expected number of targets calculated in the notebook estimated_number_of_events and the time required per target is calculated in notebook event_simulator. 

In [402]:
# Number of different categories of events per year
nhigh_mag_events = 30
nstellar_events = 10
nlongte_events = 8

In [403]:
mulens_targets = TargetSet()

In [404]:
mulens_targets.survey_years = [2026, 2027, 2028, 2029]
mulens_targets.survey_semesters = {2026: ['B'], 
                                 2027: ['A', 'B'],
                                 2028: ['A', 'B'],
                                 2029: ['A']}
mulens_targets.semester_fraction = [1.0, 1.0] # Downscaling of events for visibility already included
mulens_targets.n_planet_events_per_year = nhigh_mag_events
mulens_targets.n_stellar_events_per_year = nstellar_events
mulens_targets.n_bh_events_per_year = nlongte_events
mulens_targets.calc_total_n_events()

Expect ~30 planetary events per year, and 120.0 over the remaining survey lifetime
Expect ~10 stellar events per year, and 40.0 over the remaining survey lifetime
Expect ~8 stellar remnant events per year, and 32.0 over the remaining survey lifetime


In [405]:
# Estimates of time required in hours on the 1m/Sinistro network to observe the different categories of events
mulens_targets.sinistro_time_per_planet = 13
mulens_targets.sinistro_time_per_star = 17.0
mulens_targets.sinistro_time_per_bh = 9.0
mulens_targets.calc_sinistro_time()

Time on 1m/Sinistro network required for planetary events per year = 390hrs
Time on 1m/Sinistro network required for stellar events per year = 170.0hrs
Time on 1m/Sinistro network required for stellar remnant events per year = 72.0hrs
Total Sinistro queue time per semester = [316.0, 316.0, 316.0, 316.0, 316.0, 316.0]hrs


In [417]:
# Estimates of time required in hours on the 2m/FLOYDS network to observe the different categories of events
mulens_targets.floyds_targets_per_year = 10
mulens_targets.floyds_frac = 0.9 # Fraction of events likely to be bright enough for FLOYDS
mulens_targets.calc_floyds_time(stellar=False)

Number of stellar remnant events bright enough for FLOYDS per year = 3
Time on 2m/FLOYDS network required for stellar remnant events per year = 11hrs
Number of planetary events bright enough for FLOYDS per year = 3
Time on 2m/FLOYDS network required for planetary events per year = 11hrs
Number of stellar events bright enough for FLOYDS per year = 3
Time on 2m/FLOYDS network required for stellar events per year = 11hrs
Total time on 2m/FLOYDS in year 2026: 22
Total time on 2m/FLOYDS in year 2027: 22
Total time on 2m/FLOYDS in year 2028: 22
Total time on 2m/FLOYDS in year 2029: 22
Time on 2m/FLOYDS in semester 2026B: 22.0
Time on 2m/FLOYDS in semester 2027A: 22.0
Time on 2m/FLOYDS in semester 2027B: 22.0
Time on 2m/FLOYDS in semester 2028A: 22.0
Time on 2m/FLOYDS in semester 2028B: 22.0
Time on 2m/FLOYDS in semester 2029A: 22.0


In [407]:
# Estimate of time required in hours on the 2m/MusCAT or 2m/Spectral network to 
# observe high mag events in RAPID RESPONSE MODE
# Note 'stellar' for this function refers to all single stellar events
mulens_targets.vis_fraction_2m = 6.0 / 24.0
mulens_targets.calc_2m_imagers_time(mulens_targets.n_planet_events_per_year)

RR-mode needed for 20.0% of 30
Visibility fraction = 0.25
Total time on 2m/imagers in RR mode in year 2026: 18.0
Total time on 2m/imagers in RR mode in 2026A semester: 9.0
Total time on 2m/imagers in RR mode in 2026B semester: 9.0
Total time on 2m/imagers in RR mode in year 2027: 18.0
Total time on 2m/imagers in RR mode in 2027A semester: 9.0
Total time on 2m/imagers in RR mode in 2027B semester: 9.0
Total time on 2m/imagers in RR mode in year 2028: 18.0
Total time on 2m/imagers in RR mode in 2028A semester: 9.0
Total time on 2m/imagers in RR mode in 2028B semester: 9.0
Total time on 2m/imagers in RR mode in year 2029: 18.0
Total time on 2m/imagers in RR mode in 2029A semester: 9.0
Total time on 2m/imagers in RR mode in 2029B semester: 9.0


In [408]:
# Estimate the remaining Rapid Response time required from the 1m network
mulens_targets.vis_fraction_1m = (24.0 - 6.0) / 24.0
mulens_targets.calc_1m_rr_time(mulens_targets.n_planet_events_per_year)

Total time on 1m/imagers in RR mode in year 2026: 54.0
Total time on 1m/imagers in RR mode in 2026A semester: 27.0
Total time on 1m/imagers in RR mode in 2026B semester: 27.0
Total time on 1m/imagers in RR mode in year 2027: 54.0
Total time on 1m/imagers in RR mode in 2027A semester: 27.0
Total time on 1m/imagers in RR mode in 2027B semester: 27.0
Total time on 1m/imagers in RR mode in year 2028: 54.0
Total time on 1m/imagers in RR mode in 2028A semester: 27.0
Total time on 1m/imagers in RR mode in 2028B semester: 27.0
Total time on 1m/imagers in RR mode in year 2029: 54.0
Total time on 1m/imagers in RR mode in 2029A semester: 27.0
Total time on 1m/imagers in RR mode in 2029B semester: 27.0


In [409]:
mulens_targets.calc_time_totals()

Sinistro queue  [316.0, 316.0, 316.0, 316.0, 316.0, 316.0]
Sinistro RR  [54.0, 54.0, 54.0, 54.0, 54.0, 54.0, 54.0, 54.0]
2m RR  [18.0, 18.0, 18.0, 18.0, 18.0, 18.0, 18.0, 18.0]
FLOYDS  [68.0, 68.0, 68.0, 68.0, 68.0, 68.0]
Semester 2026B Sinistro total 316.0
 -> Sinistro queue mode 316.0
 -> Sinistro RR mode 54.0
Semester 2027A Sinistro total 316.0
 -> Sinistro queue mode 316.0
 -> Sinistro RR mode 54.0
Semester 2027B Sinistro total 316.0
 -> Sinistro queue mode 316.0
 -> Sinistro RR mode 54.0
Semester 2028A Sinistro total 316.0
 -> Sinistro queue mode 316.0
 -> Sinistro RR mode 54.0
Semester 2028B Sinistro total 316.0
 -> Sinistro queue mode 316.0
 -> Sinistro RR mode 54.0
Semester 2029A Sinistro total 316.0
 -> Sinistro queue mode 316.0
 -> Sinistro RR mode 54.0


Sinistro queue time total over project: 1896.0hrs
Sinistro queue time total over project: 432.0hrs
FLOYDS time total over project: 408.0hrs
2m imagers RR-time total over project: 144.0hrs


## Cataclysmic Variables

Dwarf novae:
Expect ~10 per night from LSST but not all will be accessible to LCO 
Expect ~3 superhump candidates from LSST per night 
From Buckley et al 2025: DNe are ~60% of all CVs, and of those LSST will detect ~20% of outbursts. 

AM CVNs: expect 10 per year from LSST

In [410]:
# AM CVns
print('AM CVns')
n_amcvns = 30 # 10/year
obs_time = 3.0 # hrs per target continuous imaging with 1min exposures
sinistro_time = n_amcvns * obs_time
print('Sinistro time: ' + str(sinistro_time) + 'hrs')

# Assuming these are evenly distributed across all semesters of the project 
amcnv_sinistro_time_per_semester = sinistro_time / 6.0 
print('Sinistro time per semester: ' + str(amcnv_sinistro_time_per_semester) + 'hrs')

AM CVns
Sinistro time: 90.0hrs
Sinistro time per semester: 15.0hrs


### Reconnaissance Observations

Since CVs have much bluer colors than most microlensing source stars, and their outbursts have a strongly color-dependent signature (brighter in the blue), reconnaissance observations of multiple filters will help to distinsguish them. 

These will be 3 groups of SDSS-u,g,r,i + Pan-STARRs-z images, spaced by 30min.  
Exposure times are estimated for a star at 15mag in r, which is the selection criteria. 

In [411]:
sinistro_overhead = 28 # seconds 
exptime = 200 # seconds
slew = 120 # seconds
nfilters = 5
nvisits = 3
single_visit = (exptime +sinistro_overhead) * nfilters + slew # seconds
recon_group = nvisits * single_visit # seconds 
print('Reconnassance observations for a single target requires ' + str(round((recon_group/3600.0),0)) + 'hrs')

n_recon = 30 # per year
total_recon_time = (n_dne * 3.0 * recon_group) / 3600.0
print('Total time for recon obs: ' + str(round(total_recon_time,0)) + 'hrs over 3 years')
recon_per_semester = total_recon_time / 6.0
print('Recon time per semester: ' + str(round(recon_per_semester,0)) + 'hrs')

Reconnassance observations for a single target requires 1.0hrs
Total time for recon obs: 94.0hrs over 3 years
Recon time per semester: 16.0hrs


### Dwarf Novae monitoring

Objective of this observing mode is to measure the periodicity that occurs during outburst, with periods ranging from 80min to several hours. 
Targets will be selected from the recon observations. 
Monitoring will consist of imaging in g,r every 15min for 24hrs. 

In [412]:
sinistro_overhead = 28 # seconds 
exptime = 200 # seconds
slew = 120 # seconds
nfilters = 2
nvisits = 24 * 4
n_dne = 10 * 3  # 10 targets per year

single_visit = (exptime + sinistro_overhead) * nfilters + slew # seconds
monitoring_time_per_target = nvisits * single_visit
print('Monitoring time per target = ' + str(round((monitoring_time_per_target/3600.0),0)) + 'hrs')

total_monitoring_time = n_dne * monitoring_time_per_target
print('Total monitoring time = ' + str(round((total_monitoring_time/3600.0),0)) + 'hrs')

dne_sinistro_time_per_semester = total_monitoring_time / (6 * 3600)
print('Monitoring time per semester = ' + str(round(dne_sinistro_time_per_semester,0)) + 'hrs per semester')

Monitoring time per target = 15.0hrs
Total monitoring time = 461.0hrs
Monitoring time per semester = 77.0hrs per semester


## Black Hole X-ray Binaries

BHXB accrete material from donor star companions.  How they do so indicates the nature of the system: Early spectral-type donor systems tend to be wind-fed, whereas in late spectral-type donors, the black hole will accrete matter from the donor via Roche lobe overflow and
an accretion disc will form around it. Material in the disk heats up radiates at multiple wavelengths during infall: long (optical) radiation from the outer disk, X-rays in the inner disk. This material can accumulate in the inner disk, causing a gradual rise in brightness over years.  Eventually, a critial mass is reached causing thermal runaway and triggering a shockwave that propagates through the disk.  This can be observed as an outburst lasting a few months, as the object suddenly brightens by ~2mag.  

From Poshak: So early triggering of XRB outbursts is very important to study the disc instability model (which predicts optical from the outer disc rising before inner X-rays). The problem is that XRBs also show significant low-level flickering activity in quiescence; LSST monitoring will likely pick up flickering in many XRB candidates, but we do not yet know how to distinguish a transient rise from true outburst start.
This is what LCO should be able to help a lot, but providing more densely sampled (daily?) monitoring for promising candidates.


Rubin's long baseline in multiple bands is well suited for monitoring the rise but is too low to characterize the outburst.  Higher cadence multi-band optical data is required to characterize the outburst, particularly during the early phases because the optical precursor signature can be used to trigger multi-wavelength observations across other facilities.  (E.g. XB-NEWS). 

These objects are more easily detected by optical surveys thanks to their wider field of view than X-ray facilies and good spatial resolution. ZTF search by Wang et al identified four known and two new objects using ZTF SDSS-g, r data; they estimate that LSST will detect 43% of outbursting systems.  

Russel et al (2019) XB-NEWS system built to identify BHB outbursts from public LCO archival data, demonstrates that LCO can perform follow-up.  

86% of BHB outbursts are expected to come from the Galactic bulge region. 
Wang et al predict LSST will detect 10% of all galactic LMXBs, i.e., roughly 400 objects.
BlackCAT catalog (Corral-Santana et al. 2016, 59 objects): ~1300 Galactic black hole transients. This means that we have only discovered less than ~5% of the total Galactic distribution.

Of the BlackCAT catalog, 35 objects are bright enough to be monitored by LCO, and LSST will add to this as the survey progresses.  
Online catalog is up to date: https://www.astro.puc.cl/BlackCAT/transients.php

Yanes-Rizo, 2026: 73 black-hole candidates in X-ray transients known to date, optical and near-infrared quiescent counterparts have only been identified for 34 objects. Out of these, confident optical or near infrared (NIR) counterparts exist for 56 sources in outburst and only 34 in quiescence.

We will monitor Rubin alerts for the catalog of known BHXBs, and add new candidates to that catalog as the survey progresses.  
If the LSST-detected rise suggests an outburst is imminent, we will monitor the target in multiple optical filters. 

ZTF identifed 6 outbursting targets in ~5yrs of ZTF data, but they only see <40% in the northern hemisphere.  
Therefore expect about 12 targets outburst over the same time period in the south.  
Over the course of a Key Project, expect ~7-8 targets. 

In the galactic plane, LSST will concentrate on griz filters with most data in iz, but still have ~week between datapoints in the same filter.  LS4 will use the same filters, and it is not clear where in the GalPlane they will observe.  It's co-survey strategy with LSST means that it will suffer the same observing gaps.  

LCO can contribute to the regular ~weekly monitoring, in one (r?) filter.  ~40 currently known targets are bright enough, though this is expeted to rise over the lifetime of the Key Project.  

When an outburst is detected, the cadence will increase to daily. 

In [413]:
# Monitoring observations
sinistro_overhead = 28 # seconds 
exptime = 300 # seconds
slew = 120 # seconds
nfilters = 1
nvisits = 52/2 * 3 # visits over Key Project (weekly for 3 years)
single_visit = (exptime + sinistro_overhead) * nfilters + slew # seconds
monitoring_time_per_target = nvisits * single_visit
print('Monitoring time per target = ' + str(round((monitoring_time_per_target/3600.0),0)) + 'hrs')

nbhxb_per_year = [40, 50, 60]  # Years 1-3
total_monitoring_time_year = []

for i,n_bhxb in enumerate(nbhxb_per_year):
    total_monitoring_time_year.append(n_bhxb * monitoring_time_per_target)
    print('Total monitoring time in year ' + str(i) + ' = ' + str(round((total_monitoring_time_year[i]/3600.0),0)) + 'hrs')

    monitoring_time_per_semester = total_monitoring_time_year[i] / 2
    print('Monitoring time per semester in year ' + str(i) + ' = ' + str(round((monitoring_time_per_semester/3600.0),0)) + 'hrs per semester')

Monitoring time per target = 10.0hrs
Total monitoring time in year 0 = 388.0hrs
Monitoring time per semester in year 0 = 194.0hrs per semester
Total monitoring time in year 1 = 485.0hrs
Monitoring time per semester in year 1 = 243.0hrs per semester
Total monitoring time in year 2 = 582.0hrs
Monitoring time per semester in year 2 = 291.0hrs per semester


In [414]:
# Outburst observations 
noutburst = 6  # Over project lifetime
nvisits = 365 # daily for 1yr
sinistro_overhead = 28 # seconds 
exptime = 300 # seconds
slew = 120 # seconds
nfilters = 4
single_visit = (exptime + sinistro_overhead) * nfilters + slew # seconds
outburst_time_per_target = nvisits * single_visit
print('Monitoring time per outburst target = ' + str(round((outburst_time_per_target/3600.0),0)) + 'hrs')

total_outburst_time = noutburst * outburst_time_per_target
print('Total monitoring time for outburst targets = ' + str(round((total_outburst_time/3600.0),0)) + 'hrs')
xrb_time_per_semester = total_outburst_time/(6 *3600.0)
print('Monitoring time for outburst targets per semester = ' + str(round(xrb_time_per_semester,0)) + 'hrs per semester')

Monitoring time per outburst target = 145.0hrs
Total monitoring time for outburst targets = 871.0hrs
Monitoring time for outburst targets per semester = 145.0hrs per semester


## Total Time Request


In [415]:
# Microlensing
mulens_targets.calc_time_totals()

Sinistro queue  [316.0, 316.0, 316.0, 316.0, 316.0, 316.0]
Sinistro RR  [54.0, 54.0, 54.0, 54.0, 54.0, 54.0, 54.0, 54.0]
2m RR  [18.0, 18.0, 18.0, 18.0, 18.0, 18.0, 18.0, 18.0]
FLOYDS  [68.0, 68.0, 68.0, 68.0, 68.0, 68.0]
Semester 2026B Sinistro total 316.0
 -> Sinistro queue mode 316.0
 -> Sinistro RR mode 54.0
Semester 2027A Sinistro total 316.0
 -> Sinistro queue mode 316.0
 -> Sinistro RR mode 54.0
Semester 2027B Sinistro total 316.0
 -> Sinistro queue mode 316.0
 -> Sinistro RR mode 54.0
Semester 2028A Sinistro total 316.0
 -> Sinistro queue mode 316.0
 -> Sinistro RR mode 54.0
Semester 2028B Sinistro total 316.0
 -> Sinistro queue mode 316.0
 -> Sinistro RR mode 54.0
Semester 2029A Sinistro total 316.0
 -> Sinistro queue mode 316.0
 -> Sinistro RR mode 54.0


Sinistro queue time total over project: 1896.0hrs
Sinistro queue time total over project: 432.0hrs
FLOYDS time total over project: 408.0hrs
2m imagers RR-time total over project: 144.0hrs


In [416]:
# Sinistro queue totals per year:
print(mulens_targets.sinistro_1m_queue[0], mulens_targets.sinistro_1m_queue[1], 
      amcnv_sinistro_time_per_semester*2, dne_sinistro_time_per_semester*2,
      recon_per_semester*2, xrb_time_per_semester)

sinistro_queue_per_year = mulens_targets.sinistro_1m_queue[0] + mulens_targets.sinistro_1m_queue[1] \
                            + amcnv_sinistro_time_per_semester*2 + dne_sinistro_time_per_semester*2 \
                            + recon_per_semester*2 + xrb_time_per_semester*2

print('Annual sinistro queue time: ' + str(round(sinistro_queue_per_year,0)) + 'hrs')

316.0 316.0 30.0 153.6 31.5 145.1888888888889
Annual sinistro queue time: 1137.0hrs
